# Citation markers in the hypothesis (v9)

The classifier has always read the citing sentence with every citation marker deleted. But 298 of
the 304 ETIQUETTE citations in train (71 of 73 in test) cite this paper **inside a group of
references**, and ETIQUETTE is 27% of grouped citations against 0.6% of single ones. This run keeps
the markers as readable tags, `[CITED]`, `[CITED, with others]` and `[other]`
(`train_nli.py --keep-markers`), and changes nothing else: same model, data, folds, seed and settings
as the current classifier (`results/nli_classifier_v1.md`).

- **Phase 1** (about 75 minutes): 3-fold cross-validation, compared citation by citation with the
  current classifier's cross-validation, overall and separately for single and grouped citations.
- **Phase 2** (about 90 minutes): 3 seeds predicting test. **It runs only if Phase 1 passes the
  gate fixed here in advance:** cross-validated macro-F1 at least 0.580 (current 0.570 + 0.01) and
  NOT ACCURATE F1 not below the current 0.513.

Everything is saved to `MyDrive/CiteAudit_runs/v9`; if Colab disconnects, Run all again and finished
work is skipped.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v9'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf

import subprocess, json, statistics, math
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
def train(name, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r --epochs 3 '
        f'--batch 2 --accum 4 --seed {seed} --amp --keep-markers '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']
line = lambda name, xs: print(f'{name:40s} ' + ' '.join(f'{x:9.3f}' for x in xs))

def compare(new, base, claims, fine, title):
    """Paired comparison: overall, by citation type, ETIQUETTE and substantive-error recall, McNemar."""
    new, base = {r['id']: r for r in new}, {r['id']: r for r in base}
    ids = sorted(new)
    grouped = {i: '<|multi_cit|>' in claims[i] for i in ids}
    print(f'\n{title}\n{"":40s} ' + ' '.join(f'{c:>9s}' for c in COLS))
    for sl, keep in [('all', lambda i: True), ('single citation', lambda i: not grouped[i]),
                     ('grouped citation', lambda i: grouped[i])]:
        sub = [i for i in ids if keep(i)]
        line(f'current, {sl} ({len(sub)})', scores([base[i] for i in sub]))
        line(f'markers, {sl} ({len(sub)})', scores([new[i] for i in sub]))
    subst = {'CONTRADICT', 'NOT_SUBSTANTIATE', 'OVERSIMPLIFY', 'MISQUOTE'}
    for name, keep in [('ETIQUETTE', lambda f: f == 'ETIQUETTE'), ('substantive errors', lambda f: f in subst)]:
        sub = [i for i in ids if keep(fine[str(i)])]
        rec = lambda d: sum(d[i]['prediction'] == 'NOT_ACCURATE' for i in sub) / max(1, len(sub))
        print(f'{name} ({len(sub)}) recalled as NOT ACCURATE: current {rec(base):.3f}, markers {rec(new):.3f}')
    fp = lambda d: sum(d[i]['prediction'] == 'NOT_ACCURATE' and d[i]['gold'] == 'ACCURATE' and grouped[i] for i in ids)
    print(f'accurate grouped citations wrongly called NOT ACCURATE: current {fp(base)}, markers {fp(new)}')
    a = sum(new[i]['prediction'] == new[i]['gold'] != base[i]['prediction'] for i in ids)
    b = sum(base[i]['prediction'] == base[i]['gold'] != new[i]['prediction'] for i in ids)
    p = min(1., 2 * sum(math.comb(a + b, j) for j in range(min(a, b) + 1)) / 2 ** (a + b)) if a + b else 1.
    print(f'right only with markers {a}, right only without {b}, McNemar exact p = {p:.4f}')

## Phase 1: 3-fold cross-validation over train

In [ ]:
for k in range(3):
    train(f'cv_markers_f{k}', 0, f'--folds 3 --fold {k}')
BASE_DIR = 'results/classification/v5'   # the current classifier's CV, same folds and seed
new = sum((read(f'{RES}/cv_markers_f{k}/oof.jsonl') for k in range(3)), [])
base = sum((read(f'{BASE_DIR}/cv_nli_f{k}/oof.jsonl') for k in range(3)), [])
claims = {c['id']: c['claim'] for c in read(f'{T}/ours_20r_train_cv/claims_train.jsonl')}
fine = json.load(open('data_train/fine_labels/train.json'))
compare(new, base, claims, fine, 'cross-validated, 2141 citations')
cv, cv_base = scores(new), scores(base)
GO = cv[1] >= 0.580 and cv[3] >= cv_base[3]
json.dump({'cv_markers': cv, 'cv_current': cv_base, 'phase2': GO}, open(f'{RES}/cv_summary.json', 'w'), indent=2)
print('\nGate passed: Phase 2 will run.' if GO else '\nGate not passed (needs macro-F1 >= 0.580 and NOT ACCURATE F1 not lower): Phase 2 is skipped.')

## Phase 2: 3 seeds predicting test (only if Phase 1 passed)

In [ ]:
if GO:
    for seed in (0, 1, 2):
        train(f'final_markers_s{seed}', seed, f'--predict test={T}/ours_20r_test')
    runs = [read(f'{RES}/final_markers_s{s}/ours_20r_test.jsonl') for s in (0, 1, 2)]
    print('test, mean ± sd over 3 seeds')
    for name, xs in zip(COLS, zip(*[scores(r) for r in runs])):
        print(f'  {name:9s} {statistics.mean(xs):.3f} ± {statistics.stdev(xs):.3f}')
    # Average the 3 seeds' probabilities, then compare with the current classifier's 3-seed average.
    def average(runs):
        out = []
        for rs in zip(*[sorted(r, key=lambda x: x['id']) for r in runs]):
            p = {c: statistics.mean(r['probs'][c] for r in rs) for c in LABELS}
            out.append({**rs[0], 'probs': p, 'prediction': max(p, key=p.get)})
        return out
    cur = average([read(f'results/classification/v5/final_nli_s{s}/ours_20r_test.jsonl') for s in (0, 1, 2)])
    tclaims = {c['id']: c['claim'] for c in read(f'{T}/ours_20r_test/claims_test.jsonl')}
    tfine = json.load(open('data_train/fine_labels/test.json'))
    compare(average(runs), cur, tclaims, tfine, 'test, 606 citations, 3 seeds averaged')

## Download

Zips `MyDrive/CiteAudit_runs/v9` (predictions and logs, no model weights). Put `v9.zip` in the
repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v9.zip v9 -x '*.part/*'
from google.colab import files
files.download('/content/v9.zip')